* **Formula:**
  $$\text{Mom}_{21} = \frac{P_{t}}{P_{t-21}} - 1$$

.
├── core
│   ├── analyzer.py
│   ├── auditor.py
│   ├── builder.py
│   ├── contracts.py
│   ├── engine.py
│   ├── environment.py
│   ├── features.py
│   ├── logic.py
│   ├── paths.py
│   ├── performance.py
│   ├── quant.py
│   ├── result.py
│   ├── settings.py
│   └── utils.py
└── strategy
    └── registry.py

In its current form, **no**, simply swapping `iloc` for `first()` inside this specific function will not make it significantly faster. In fact, `first()` is a method designed for `GroupBy` objects; calling it on a Series is often just a wrapper for similar logic.

However, **the way this function is likely being used (via `.apply()` or a loop) is the real performance bottleneck.**

Here is a breakdown of why this code is slow and how to make it "Pandas-fast."

### 1. The Bottlenecks in your current code:
1.  **`data.apply(...)`**: If you pass a DataFrame with 5 columns, it runs the loop 5 times. If you have a MultiIndex and call this inside a `groupby().apply()`, it runs thousands of times. Python "Apply" is essentially a hidden `for` loop.
2.  **`data.dropna()`**: This creates a complete copy of the data in memory just to find two numbers.
3.  **Recursion/Conditionals**: Every time this function is called, it performs multiple `if` checks and type checks. Across 10,000 tickers, this overhead adds up.

### 2. How to achieve "Vectorized" speed (The faster way)
Instead of calling a function for every ticker, you should process the entire `df_decision` (all 17,000+ rows) at once using **Grouped Vectorization**.

**If you replace your function logic with this, it will be roughly 50x–100x faster:**

```python
def calculate_gain_vectorized(df, min_points=2):
    # 1. Group by Ticker (level 0) and get first/last valid values
    # This happens at C-speed, skipping NaNs automatically
    grouped = df.groupby(level=0)
    
    first_vals = grouped.first()
    last_vals = grouped.last()
    counts = grouped.count()

    # 2. Vectorized Log Return: ln(last / first)
    # This handles all tickers and all columns simultaneously
    with np.errstate(divide='ignore', invalid='ignore'):
        gain = np.log(last_vals / first_vals)
    
    # 3. Handle your business logic (min_points and non-positive prices)
    # Set to -10.0 if prices are <= 0 (matches your logic)
    gain[(first_vals <= 0) | (last_vals <= 0)] = -10.0
    
    # Set to 0.0 if not enough points
    gain[counts < min_points] = 0.0
    
    return gain.fillna(0.0)
```

### 3. Comparison of Approaches

| Method | Speed | Why? |
| :--- | :--- | :--- |
| **Your Function (`.apply`)** | **Slow** | Loops in Python; handles one Series at a time; high overhead. |
| **`iloc` vs `first` (inside Series)** | **Neutral** | Minimal difference when operating on a single pre-sliced Series. |
| **Vectorized (`groupby().first()`)** | **Fastest** | Operations happen in C/NumPy across the whole block of 17k rows at once. |

### Summary Recommendation
If you only have 17,000 rows and you only run this once, your current code is fine. 

**But**, if you are running this inside a **backtest loop** (e.g., repeating this calculation thousands of times for different dates), you should **delete the `.apply()` logic** and use the **Grouped Vectorization** approach. It will turn a process that takes seconds into one that takes milliseconds.

In `stocks/notebooks_RLVR/core/features.py`, `IR_63` is the **63-day rolling Information Ratio**. 

The calculation is performed by `QuantUtils.calculate_rolling_ir` in `stocks/notebooks_RLVR/core/quant.py` using the following logic:

1.  **Active Returns**: It first calculates the daily active returns by subtracting the benchmark (market) returns from the ticker's daily returns.
    $$Active\_Ret = R_{ticker} - R_{benchmark}$$
2.  **Rolling Mean ($\mu$)**: It calculates the 63-day rolling mean of these active returns.
3.  **Rolling Volatility ($\sigma$)**: It calculates the 63-day rolling standard deviation of these active returns.
4.  **Final Ratio**: The result is the mean divided by the standard deviation:
    $$IR_{63} = \frac{\mu_{active}}{\sigma_{active}}$$

Note that this is calculated on a **daily scale** and is not annualized. A small epsilon ($10^{-8}$) is used in the denominator to prevent division by zero.

Consider two stocks, **Stock A (High Price)** and **Stock B (Low Price)**, both moving up by $0.25 each day for 5 days.

### 1. Data Setup
| Day | Stock A (Price) | Stock B (Price) |
| :--- | :--- | :--- |
| t-4 | $1,000.00 | $5.00 |
| t-3 | $1,000.25 | $5.25 |
| t-2 | $1,000.50 | $5.50 |
| t-1 | $1,000.75 | $5.75 |
| **t (Today)** | **$1,001.00** | **$6.00** |

---

### 2. Calculate Raw Slope
Using the derived OLS formula: $\text{Slope} = \frac{2y_t + 1y_{t-1} + 0y_{t-2} - 1y_{t-3} - 2y_{t-4}}{10}$

**For Stock A:**
$$\text{Slope} = \frac{2(1001) + 1(1000.75) + 0 - 1(1000.25) - 2(1000)}{10}$$
$$\text{Slope} = \frac{2002 + 1000.75 - 1000.25 - 2000}{10} = \frac{2.5}{10} = \mathbf{0.25}$$

**For Stock B:**
$$\text{Slope} = \frac{2(6) + 1(5.75) + 0 - 1(5.25) - 2(5)}{10}$$
$$\text{Slope} = \frac{12 + 5.75 - 5.25 - 10}{10} = \frac{2.5}{10} = \mathbf{0.25}$$

**Result:** Both have an identical slope of **0.25**. To a computer looking at "Raw Slope," these two stocks look exactly the same.

---

### 3. The "Reality" Problem
*   **Stock A** moved from $1000 to $1001. That is a **0.1%** gain. It is effectively "flat."
*   **Stock B** moved from $5 to $6. That is a **20%** gain. It is "mooning."

If you used these raw slopes to rank stocks, the computer would treat a boring utility stock and a high-growth tech stock as having the same momentum just because their dollar-change was the same.

---

### 4. The Solution: Log Slope (`Slope_P_5`)
If we take the `LN` (Natural Log) of the prices first:
*   **Stock A Log Slope:** $\approx \mathbf{0.00025}$ (reflecting the tiny 0.025% daily compounded growth).
*   **Stock B Log Slope:** $\approx \mathbf{0.04500}$ (reflecting the massive 4.5% daily compounded growth).

**By using `LN`, Stock B's slope becomes 180x larger than Stock A's**, correctly identifying it as the stock with actual momentum.

The derivation relies on centering the time index to simplify the Ordinary Least Squares (OLS) formula.

For a 5-day window, let the time indices ($x$) be centered around the middle of the window:
$$x = \{-2, -1, 0, 1, 2\}$$

### 1. The General OLS Slope Formula
The slope $m$ is given by:
$$m = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}$$

### 2. Simplify the Denominator
First, calculate the mean of $x$ ($\bar{x}$):
$$\bar{x} = \frac{-2 - 1 + 0 + 1 + 2}{5} = 0$$

Since $\bar{x} = 0$, the denominator simplifies to $\sum x_i^2$:
$$\sum x_i^2 = (-2)^2 + (-1)^2 + 0^2 + 1^2 + 2^2$$
$$\sum x_i^2 = 4 + 1 + 0 + 1 + 4 = \mathbf{10}$$

### 3. Simplify the Numerator
Using the property $\sum (x_i - \bar{x})(y_i - \bar{y}) = \sum x_i y_i - \bar{x}\sum y_i$, and since $\bar{x} = 0$:
$$\text{Numerator} = \sum x_i y_i$$

Expanding this for our 5 indices (where $y_t$ is the most recent value):
$$\sum x_i y_i = (2 \cdot y_t) + (1 \cdot y_{t-1}) + (0 \cdot y_{t-2}) + (-1 \cdot y_{t-3}) + (-2 \cdot y_{t-4})$$

### 4. Final Result
Combining the numerator and denominator:
$$m = \frac{2y_t + 1y_{t-1} + 0y_{t-2} - 1y_{t-3} - 2y_{t-4}}{10}$$

### Code Implementation
This is why the code in `core/quant.py` uses those specific weights:
```python
return (
    2 * series             # y_t (today)
    + 1 * series.shift(1)  # y_t-1
    + 0 * series.shift(2)  # y_t-2
    + -1 * series.shift(3) # y_t-3
    + -2 * series.shift(4) # y_t-4
) / 10.0
```

$$\text{Slope} = \frac{(2 \cdot y_t) + (1 \cdot y_{t-1}) + (0 \cdot y_{t-2}) + (-1 \cdot y_{t-3}) + (-2 \cdot y_{t-4})}{10}$$

Based on the code in `core/quant.py` and the data from `capture.png`, the first two values of ATR are calculated as follows:

### 1. Calculation Logic
The code uses **Wilder's Smoothing** (Exponential Moving Average with $\alpha = 1/\text{period}$) applied to the **True Range (TR)**.

*   **ATR Period:** 14 (defined in `core/settings.py`).
*   **True Range (TR):** $\max(\text{High}_t - \text{Low}_t, |\text{High}_t - \text{Close}_{t-1}|, |\text{Low}_t - \text{Close}_{t-1}|)$.
*   **Smoothing Formula:** $ATR_t = \alpha \cdot TR_t + (1 - \alpha) \cdot ATR_{t-1}$.

### 2. The First Two Values
Because the code uses `close.shift(1)` to find the previous close, the calculation is delayed by one row:

#### **Value 1: January 3, 2024 (Index 1)**
The very first row (Jan 2) has no "previous close," so its TR is `NaN`. The code's `ewm(adjust=False)` logic treats the first valid TR as the starting ATR value.
1.  **Calculate TR for Jan 3:**
    $$TR_{Jan3} = \max(183.969 - 181.544, |183.969 - \text{Close}_{Jan2}|, |181.544 - \text{Close}_{Jan2}|)$$
2.  **First ATR Value:**
    $$ATR_{Jan3} = TR_{Jan3}$$

#### **Value 2: January 4, 2024 (Index 2)**
The second value uses the recursive Wilder's formula with $\alpha = 1/14 \approx 0.0714$.
1.  **Calculate TR for Jan 4:**
    $$TR_{Jan4} = \max(181.208 - 179.020, |181.208 - \text{Close}_{Jan3}|, |179.020 - \text{Close}_{Jan3}|)$$
2.  **Second ATR Value:**
    $$ATR_{Jan4} = \left(\frac{1}{14} \cdot TR_{Jan4}\right) + \left(\frac{13}{14} \cdot ATR_{Jan3}\right)$$

### Summary for your Data:
| Date | High | Low | TR Calculation | ATR Value |
| :--- | :--- | :--- | :--- | :--- |
| **1/2/2024** | 186.503 | 181.999 | `NaN` (No prev close) | `NaN` |
| **1/3/2024** | 183.969 | 181.544 | First valid TR | **Value 1: $TR_{Jan3}$** |
| **1/4/2024** | 181.208 | 179.020 | Second valid TR | **Value 2: $0.0714 \cdot TR_{Jan4} + 0.9286 \cdot ATR_{Jan3}$** |

*Note: The "Adj Close" values (not visible in your image but present in `df_ohlcv`) are required to calculate the exact True Range if the gap between days is larger than the daily High-Low range.*

To verify all 18 columns in your `subset_df`, here is the breakdown of the calculations you can replicate in Excel using the `raw_price_audit.csv` and `raw_macro_audit.csv` files.

### 📊 Excel Verification Guide for `features_df`

| # | Column | Excel Formula / Logic |
| :--- | :--- | :--- |
| **0** | **ATR** | `Average(True Range, 14)` (or your `atr_period`). $TR = \max(H-L, |H-C_{prev}|, |L-C_{prev}|)$ |
| **1** | **ATRP** | `ATR / Adj Close` (Normalized volatility as a percentage of price). |
| **2** | **TRP** | `True Range / Adj Close` (Daily movement potential as a percentage). |
| **3** | **RSI** | Standard Wilder's RSI (usually 14 periods) using `Adj Close`. |
| **4** | **Mom_21** | `(Price_t / Price_{t-21}) - 1` (21-day simple return). |
| **5** | **Consistency** | `CountIf(Returns > 0, 5) / 5` (Percentage of up-days in the last 5 days). |
| **6** | **IR_63** | `Average(Active_Ret, 63) / StDev(Active_Ret, 63)`. $Active\_Ret = Ticker\_Ret - Mkt\_Ret$. |
| **7** | **Beta_63** | `COVARIANCE.P(Ticker_Ret_63, Mkt_Ret_63) / VAR.P(Mkt_Ret_63)`. |
| **8** | **DD_21** | `(Price / Max(Price, 21)) - 1` (Drawdown from 21-day high). |
| **9** | **AutoCorr_15** | `CORREL(Returns_15, Returns_15_lag1)` (Correlation of returns vs. yesterday's returns). |
| **10** | **Ret_1d** | `(Adj Close_t / Adj Close_{t-1}) - 1`. |
| **11** | **Range_Pos_20** | `(Close - Min(Low, 20)) / (Max(High, 20) - Min(Low, 20))`. |
| **12** | **Slope_P_5** | Slope of the linear regression of `LN(Adj Close)` over the last 5 days. |
| **13** | **Slope_V_5** | Slope of the linear regression of `On-Balance Volume (OBV)` over the last 5 days. |
| **14** | **Convexity** | `Slope_P_5_t - Slope_P_5_{t-1}` (The "acceleration" or change in price growth). |
| **15** | **RollingStalePct** | `CountIf(Stale_Days, 252) / 252`. Stale = (Volume=0) or (High=Low). |
| **16** | **RollMedDollarVol**| `MEDIAN(Adj Close * Volume, 252)` (Typical daily liquidity). |
| **17** | **RollingSameVol** | `CountIf(Volume_t == Volume_{t-1}, 252)` (Detects low-quality/frozen data). |

